# Build the verified dataset on Kaggle

Same script, same gate, same 582 inputs as the Colab notebook. Kaggle is here
for one reason: a 9-12 hour session that does not drop, so the run is not
restarted five times.

**Before running anything — Settings, right-hand panel:**

| setting | value | why |
|---|---|---|
| Accelerator | **GPU T4 x2** | the script uses one; the second is idle |
| Internet | **On** | the clone and the model download both need it |
| Persistence | Files only | keeps `/kaggle/working` between sessions |

Internet off is the expensive mistake: the download fails at load time, after
the session has already started spending the weekly budget.

**Nothing needs uploading.** The 582 functions come with the clone (they are
committed as `inputs.jsonl`, 246 KB) and the weights come from the Hub.

## Running it unattended

**Save Version → Save & Run All (Commit).** The notebook runs top to bottom on
Kaggle's machines with no browser open, up to twelve hours, and the result lands
in that version's Output tab. That is the whole point of being here rather than
on Colab.

Two things follow from batch mode, and both are already handled:

* **No stdin.** Anything that asks a question raises
  `StdinNotImplementedError` the moment it runs. There are no prompts left.
* **No kernel restarts.** A notebook that needs a human to press restart cannot
  run unattended - see the note after the install cell for why none is needed.

Interactive first is still the better order: run the cells by hand as far as the
twenty-function pilot, read the yield, and only then Save & Run All for the rest.


## 1. Caches to /tmp, before any import

`/kaggle/working` is a hard 20 GB and it is the only thing that survives the
session. `/tmp` is ~60 GB and is wiped. A 3 GB model download landing on the
persistent disk is the usual way that 20 GB disappears, and by the time it
matters the download has already happened.

This must be the first cell, above every import.

In [ ]:
import os

os.environ["HF_HOME"] = "/tmp/hf_cache"
os.environ["HF_DATASETS_CACHE"] = "/tmp/hf_datasets"
os.environ["TRANSFORMERS_CACHE"] = "/tmp/hf_cache"
print({k: v for k, v in os.environ.items() if k.startswith(("HF_", "TRANSFORMERS_"))})

In [ ]:
# Kaggle preinstalls its own transformers and peft, older than this needs.
# Nothing is imported in this cell - not even to check a version - because an
# already-imported package keeps winning over the one just installed. The next
# cell is therefore the first import in the notebook, which is the state a
# manual restart would have produced.
!pip install -q -U transformers peft accelerate
print("installed; nothing imported yet, so the next cell gets these versions")

## No restart needed here, and that is on purpose

The usual Kaggle rule is: install, restart, then import. The restart exists
because an already-imported preinstalled package keeps winning over the one you
just installed.

Nothing above imports anything. The environment cell touches only `os`, and the
install cell was written to import nothing at all - not even to print a version.
So the first import of `transformers` in this notebook happens *after* the
install, which is the state the restart was there to produce.

That matters because **Save & Run All has no way to restart a kernel**. A
notebook that needs a human to press restart cannot run unattended, which is the
whole reason for using Kaggle here.

In [ ]:
import os

# Set again in case this cell is being re-run after a manual restart. Harmless
# when it is not - and if it is skipped after a restart, the 3GB download lands
# on the 20GB persistent disk instead of the 60GB scratch one.
os.environ["HF_HOME"] = "/tmp/hf_cache"
os.environ["HF_DATASETS_CACHE"] = "/tmp/hf_datasets"
os.environ["TRANSFORMERS_CACHE"] = "/tmp/hf_cache"

import torch
import transformers

# Paths, not only versions. A path under Kaggle's system site-packages means the
# install did not win, and the fix is a restart before anything else.
for module in (torch, transformers):
    print(f"{module.__name__:>14} {module.__version__:>12}  {module.__file__}")

assert torch.cuda.is_available(), "Settings -> Accelerator -> GPU T4 x2"
print("cuda:", torch.cuda.get_device_name(0))

In [ ]:
!g++ --version | head -1   # the gate compiles and runs every candidate

## 2. The code and the inputs

Cloned rather than uploaded, so the gate deciding which rows exist is the one
with tests behind it, and `inputs.jsonl` arrives with it.

In [ ]:
import os
import subprocess

REPO = "safi892/fyp_training"
BRANCH = "language"
CHECKOUT = "/kaggle/working/fyp"

# No prompt: the repo clones without credentials. Set TOKEN to a personal access
# token only if that stops being true - an interactive getpass inside a notebook
# is a place to get stuck for no gain, and the masked dots make an empty answer
# look like a failed one.
TOKEN = ""

url = (f"https://{TOKEN}@github.com/{REPO}.git" if TOKEN
       else f"https://github.com/{REPO}.git")

# Never stand in the directory being removed: a process whose working directory
# is gone cannot run git, and the clone then fails with something that reads
# like a permissions problem.
os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", CHECKOUT], check=True)

done = subprocess.run(["git", "clone", "-b", BRANCH, url, CHECKOUT],
                      capture_output=True, text=True)
if done.returncode != 0:
    detail = done.stderr.replace(TOKEN, "***") if TOKEN else done.stderr
    raise SystemExit(f"clone failed:\n{detail}")

os.chdir(CHECKOUT)
print("cloned at", subprocess.run(["git", "log", "--oneline", "-1"],
                                  capture_output=True, text=True).stdout.strip())

In [ ]:
import sys

sys.path.insert(0, f"{CHECKOUT}/src")
sys.path.insert(0, f"{CHECKOUT}/scripts")
from pathlib import Path

from build_optimize_dataset import drivable_recursive, extract_candidate, judge

INPUTS = f"{CHECKOUT}/my_data_annotation/recursion_optimization/inputs.jsonl"
BASE = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
ADAPTER = None                          # base model: nothing to upload
OUT = "/kaggle/working/verified.jsonl"  # survives the session and lands in the output

print(len(drivable_recursive(Path(INPUTS), 40)), "functions   (expected 582)")

# The gate, on cases whose answer is known, before any GPU time is spent. If a
# wrong rewrite is not rejected here, nothing this notebook produces is worth
# keeping.
rec = "int fact(int n){ if(n<=1) return 1; return n*fact(n-1); }"
itr = "int fact(int n){ int r=1; for(int i=2;i<=n;i++) r*=i; return r; }"
assert judge(rec, itr, 10.0) is None
assert judge(rec, itr.replace("r=1", "r=0"), 10.0) == "different output"
assert judge(rec, rec, 10.0) == "still recursive"
assert extract_candidate("```cpp\nint f(){return 1;}\n```") == "int f(){return 1;}"
print("gate and extractor: ok")

## 3. Twenty functions first

The yield decides whether the rest is worth the hours. Rows kept ÷ 20 is what
582 will give. The fine-tune measured **8%** on CPU; anything clearly above that
means the base model is the better proposer, which is itself a result — it was
the fine-tune that was trained on 83% of targets that left the recursion in.

In [ ]:
def build_cmd(limit, samples=16, temperature=0.9, batch=4):
    adapter = f"--adapter {ADAPTER}" if ADAPTER else ""
    return (
        f"cd {CHECKOUT} && PYTHONPATH=src "
        f"python scripts/build_optimize_dataset.py "
        f"--backend hf --base {BASE} {adapter} --batch {batch} "
        f"--corpus {INPUTS} --out {OUT} "
        f"--limit {limit} --samples {samples} --temperature {temperature}"
    )

print(build_cmd(20))

Twenty first, then the rest. Both cells run in an unattended pass, and the
second skips everything the first already did - so the pilot costs nothing
except the chance to stop after reading it.

In [ ]:
!{build_cmd(20)}

## 4. The rest

One run, because the session holds. `--limit` skips what is already done, so if
it does stop, re-running this cell continues rather than repeats — and unlike
Colab's `/content`, `/kaggle/working` survives.

Watch the weekly GPU budget: roughly 30 hours, and this should want two to five.

In [ ]:
!{build_cmd(582)}

## 5. Read every row, then take it home

The gate is re-run here in front of you. These rows were written by a process
that could have been interrupted mid-line, and the whole claim of this dataset
is that every row was executed.

In [ ]:
import json

rows = [json.loads(line) for line in open(OUT) if line.strip()]
print(f"{len(rows)} verified rows\n")

for n, row in enumerate(rows, 1):
    problem = judge(row["code"], row["improved_code"], 10.0)
    print("=" * 78)
    print(f"[{n}/{len(rows)}]   re-check: {problem or 'PASSES - compiles, runs, identical output'}")
    print("-" * 78)
    print("RECURSIVE (the author's own code)")
    print(row["code"].rstrip())
    print("-" * 78)
    print("REWRITTEN (kept only because it passed)")
    print(row["improved_code"].rstrip())
    print()

In [ ]:
# The same rows as JSONL, for copy-paste into
# my_data_annotation/recursion_optimization/verified.jsonl on the laptop.
print(f"# {len(rows)} verified rows")
for row in rows:
    print(json.dumps(row, ensure_ascii=False))

### Getting the file

`/kaggle/working/verified.jsonl` appears in the **Output** tab on the right —
download it from there, or Save Version and take it from the version's output.
The printed JSONL above is the fallback, and it is saved inside the notebook.

Keep `/kaggle/working` tidy: the clone is a few hundred files against a ~500
file cap on the output directory, so remove it before saving a version if the
save complains.

In [ ]:
!du -sh /kaggle/working/* | sort -h | tail -5
!echo "files in working:" && find /kaggle/working -type f | wc -l